# 复利时光：红利低波公开方法的部分复现

**研究版本，不是原作者完整策略或源码。** 已确认 RSI(14)、250 日均线、KDJ(9,3,3) 等指标和独立 RSI 阈值方法；综合评分的完整特征及权重尚未公开确认，不生成冒充原版的评分。

阅读顺序：真实行情 → 对照作者午间快照 → 复权敏感性 → RSI 阈值事件研究 → 本地指标图。

来源及限制见 `research/fuli_time/README.md`。以 2026-11-06 为固定终点，避免重跑时研究边界变化。

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import akshare as ak
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from trading_analysis.studies.dividend_low_vol import compute_indicators, grade_score

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
REFRESH = False  # True 重新获取；默认复用已保存的真实行情
END_DATE = "20261106"
RENAME = {"日期":"date", "开盘":"open", "最高":"high", "最低":"low", "收盘":"close", "成交量":"volume"}

def load_prices(adjust):
    path = ROOT / "data" / "raw" / f"512890_daily_em_{adjust or 'none'}.csv"
    if REFRESH or not path.exists():
        raw = ak.fund_etf_hist_em(symbol="512890", period="daily", start_date="20190101", end_date=END_DATE, adjust=adjust)
        if raw.empty:
            raise ValueError("行情为空")
        path.parent.mkdir(parents=True, exist_ok=True)
        raw.to_csv(path, index=False)
    prices = pd.read_csv(path).rename(columns=RENAME)
    prices["date"] = pd.to_datetime(prices["date"])
    return prices.loc[prices.date <= pd.Timestamp(END_DATE), list(RENAME.values())].reset_index(drop=True)

prices = {"不复权": load_prices(""), "前复权": load_prices("qfq")}
indicators = {name: compute_indicators(frame) for name, frame in prices.items()}
observations = pd.read_json(ROOT / "research/fuli_time/observations.json")
print(f"ETF 512890 | AkShare {ak.__version__} | {len(prices['前复权'])} 个交易日")
display(prices["前复权"].tail(3))


ETF 512890 | AkShare 1.18.88 | 1889 个交易日


,date,open,high,low,close,volume
1886,2026-11-04,1.236,1.247,1.231,1.232,7349355
1887,2026-11-05,1.230,1.231,1.209,1.221,9835807
1888,2026-11-06,1.221,1.237,1.214,1.237,5558917


## 1. 对照作者午间快照

只比较依赖收盘序列的 RSI 和 MA250：截取该日及之前数据，将**当日最后价替换为作者已公布的 11:30 价格**。这是对公式和标的的诊断，不是独立预测。当天完整 high/low/volume 不用于此项对比。

为满足行情校验，替换价格若超过当日日线高低值，则扩展边界；不据此验证 KDJ。更早日期的前复权价格可能不同，因此该诊断选用不复权序列；前复权敏感性单独检验。


In [2]:
comparison = []
for row in observations.itertuples():
    frame = prices["不复权"].loc[prices["不复权"].date <= row.date].copy()
    if frame.empty or frame.iloc[-1].date != row.date:
        continue
    i = frame.index[-1]
    frame.loc[i, "close"] = row.observed_price
    frame.loc[i, "high"] = max(frame.loc[i, "high"], row.observed_price)
    frame.loc[i, "low"] = min(frame.loc[i, "low"], row.observed_price)
    local = compute_indicators(frame).iloc[-1]
    comparison.append({"date": row.date, "author_rsi":row.observed_rsi14, "local_rsi":local.rsi14,
                       "rsi_error":local.rsi14-row.observed_rsi14,
                       "author_bias":row.observed_bias_pct,"local_bias":local.ma250_bias_pct,
                       "bias_error":local.ma250_bias_pct-row.observed_bias_pct})
comparison = pd.DataFrame(comparison).sort_values("date")
display(comparison.round(4))
comparison.to_csv(ROOT / "outputs/fuli_time_indicator_comparison.csv", index=False)
print("最大绝对误差：", comparison[["rsi_error","bias_error"]].abs().max().to_dict())


,date,author_rsi,local_rsi,rsi_error,author_bias,local_bias,bias_error
9,2026-09-07,54.4,54.3997,-0.0003,1.32,1.3201,0.0001
8,2026-09-29,45.7,45.7400,0.0400,0.53,0.5271,-0.0029
7,2026-10-12,66.2,66.1814,-0.0186,4.12,4.1156,-0.0044
6,2026-10-14,66.9,66.8964,-0.0036,4.74,4.7434,0.0034
5,2026-10-26,49.7,49.7170,0.0170,2.19,2.1891,-0.0009
4,2026-11-02,61.0,60.9900,-0.0100,5.08,5.0752,-0.0048
3,2026-11-03,56.6,56.5993,-0.0007,4.30,4.3016,0.0016
2,2026-11-04,61.8,61.7630,-0.0370,5.30,5.3035,0.0035
1,2026-11-05,50.5,50.5511,0.0511,3.26,3.2648,0.0048
0,2026-11-06,55.1,55.1323,0.0323,4.19,4.1854,-0.0046


最大绝对误差： {'rsi_error': 0.05110914688427215, 'bias_error': 0.004844549830544942}


## 2. 已公布评分与版本差异

这些是人工记录的作者评分，**不是本地计算结果**。11 月 6 日图 6 的分档为 25/40/55/70；早期文字出现不一致，不覆盖原始记录。

In [3]:
grades = observations[["date", "published_score", "published_grade"]].copy()
grades["grade_using_nov6_image"] = grades.published_score.map(grade_score)
grades["consistent"] = grades.published_grade == grades.grade_using_nov6_image
display(grades)


,date,published_score,published_grade,grade_using_nov6_image,consistent
0,2026-11-06,32.0,D,D,True
1,2026-11-05,41.0,C,C,True
2,2026-11-04,15.0,E,E,True
3,2026-11-03,28.0,D,D,True
4,2026-11-02,16.0,E,E,True
5,2026-10-26,52.0,C,C,True
6,2026-10-14,15.0,E,E,True
7,2026-10-12,19.0,E,E,True
8,2026-09-29,75.2,B,A,False
9,2026-09-07,57.0,C,B,False


## 3. RSI <30 / >70 与未来 60 日：复权敏感性

作者方法笔记明确 RSI(14)<30 作为低位条件，>70 作为止盈条件。这里分别生成信号；低位事件定义为每次进入超卖区的第一天。

统计截止 2026-08-07（与方法笔记边界一致），只保留该日之前已走满 60 个交易日的事件。未来收益使用 `P[t+60]/P[t]-1`，这是**条件收益统计，不是实际可成交回测**：收盘信号不能保证按同一收盘价成交，也未扣手续费。事件可能重叠。前复权与不复权分别计算，不混用信号与收益口径；前复权是当前下载版本，未确认作者的历史复权版本。

作者声称：1828 个交易日、35 个超卖日、10 个样本、60 日胜率 100%、平均收益 8.42%。以下数据用于核验，不能直接采信这一说法。初始化、事件去重、持有期定义和数据版本均可能影响结果。

**不复权跨期价格变化会受拆分、分红等除权事件影响，不能直接当作投资收益。该列仅作为数据口径诊断；前复权也不代表含手续费、滑点及仓位约束的真实账户收益。**

In [4]:
study_end = pd.Timestamp("2026-08-07")
summaries, events = [], {}
for name, full in indicators.items():
    data = full.loc[full.date <= study_end].copy()
    below = data.rsi14 < 30
    entered = below & ~below.shift(1, fill_value=False)
    above = data.rsi14 > 70
    exited = above & ~above.shift(1, fill_value=False)
    # 标签只在研究终点前已有完整后续行情时可用，不把未成熟标签当作0。
    data["forward_60d_pct"] = (data.close.shift(-60) / data.close - 1) * 100
    table = data.loc[entered, ["date", "close", "rsi14", "forward_60d_pct"]].copy()
    events[name] = table
    mature = table.forward_60d_pct.dropna()
    summaries.append({"口径":name,"交易日":len(data),"超卖日":int(below.sum()),
                      "进入超卖次数":int(entered.sum()),"进入超买次数":int(exited.sum()),
                      "成熟60日样本":len(mature),"正收益比例_pct":(mature>0).mean()*100,
                      "平均60日收益_pct":mature.mean(),"最差60日收益_pct":mature.min()})
summary = pd.DataFrame(summaries)
display(summary.round(3), events["前复权"].round(4))
summary.to_csv(ROOT / "outputs/fuli_time_rsi_sensitivity.csv", index=False)


,口径,交易日,超卖日,进入超卖次数,进入超买次数,成熟60日样本,正收益比例_pct,平均60日收益_pct,最差60日收益_pct
0,不复权,1830,64,16,34,13,92.308,2.787,-46.906
1,前复权,1830,34,14,36,11,100.000,7.319,3.209


,date,close,rsi14,forward_60d_pct
70,2019-05-09,0.542,29.1918,3.6900
248,2020-02-03,0.524,15.7267,3.8168
281,2020-03-19,0.522,29.0201,8.8123
494,2021-01-29,0.642,29.9901,13.2399
611,2021-07-27,0.711,27.9274,6.1885
762,2022-03-15,0.747,27.7406,14.0562
911,2022-10-26,0.803,28.8386,4.2341
913,2022-10-28,0.792,26.6240,6.0606
1367,2024-09-06,0.982,29.2776,13.9511
1625,2025-09-30,1.139,29.2051,3.2485


## 4. 本地基础指标图

日收盘版本使用前复权 512890 行情，不等同于作者午间发布值。标记为条件触发，未推算未公开的综合评分。RSI 阈值30/70来自方法笔记；它与最新看板中的五档强弱标签不同。

In [5]:
d = indicators["前复权"]
low = d.rsi14 < 30
entry = low & ~low.shift(1, fill_value=False)
high = d.rsi14 > 70
exit_signal = high & ~high.shift(1, fill_value=False)
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[0.55,0.25,0.20], vertical_spacing=0.06)
fig.add_trace(go.Scatter(x=d.date,y=d.close,name="512890 前复权收盘"),row=1,col=1)
fig.add_trace(go.Scatter(x=d.date,y=d.ma250,name="MA250",line=dict(dash="dash")),row=1,col=1)
for mask,label,color,symbol in [(entry,"进入 RSI<30","#159b76","triangle-up"),(exit_signal,"进入 RSI>70","#e14b4b","triangle-down")]:
    fig.add_trace(go.Scatter(x=d.loc[mask,"date"], y=d.loc[mask,"close"],mode="markers",name=label,marker=dict(color=color,symbol=symbol,size=9)),row=1,col=1)
fig.add_trace(go.Scatter(x=d.date,y=d.rsi14,name="Wilder RSI14"),row=2,col=1)
fig.add_hline(y=30,line_dash="dot",row=2,col=1)
fig.add_hline(y=70,line_dash="dot",row=2,col=1)
fig.add_trace(go.Scatter(x=d.date,y=d.ma250_bias_pct,name="年线偏离 %"),row=3,col=1)
fig.add_hline(y=0,line_dash="dot",row=3,col=1)
fig.update_layout(title="红利低波公开规则 · 部分复现（非原作者综合评分）",template="plotly_white",height=850)
fig.update_yaxes(title_text="RSI",row=2,col=1)
fig.update_yaxes(title_text="偏离 %",row=3,col=1)
fig.write_html(ROOT / "outputs/fuli_time_reproduction.html",include_plotlyjs=True)
fig.show(renderer="plotly_mimetype")


## 未完成的部分

完整综合评分约有20项指标。公开说明仅给出“历史未来60日收益均值 → 按预测能力加权 → 历史百分位”的框架；全部特征、分箱、权重、训练窗口、同分容差和拥挤度定义仍不明确。未输出自创评分冒充原版。

若后续实现候选评分模型，必须采用时间顺序验证，训练时只使用已成熟的60日标签；不能在全历史训练后再把同一历史的拟合收益当作样本外效果。